In [4]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [7]:
"""
End-to-end pipeline for the House Price Prediction project.
This script contains the exact same logic that goes into the Jupyter
notebook (house_price_model.ipynb) — kept here as a plain .py so it can be
run/debugged quickly, then copied cell-by-cell into the notebook.
"""
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

RANDOM_STATE = 42
OUT_DIR = "."

# ---------------------------------------------------------------------------
# 2.1 Load & Inspect
# ---------------------------------------------------------------------------
df = pd.read_excel("data/house_prices.xlsx")
print("shape:", df.shape)
print(df.dtypes)
print(df.isna().mean().sort_values(ascending=False))

# ---------------------------------------------------------------------------
# 2.3 Cleaning & Feature Engineering
# ---------------------------------------------------------------------------


def parse_amount(x):
    """'42 Lac' -> 4_200_000.0   '1.2 Cr' -> 12_000_000.0   'Call for Price' -> None"""
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", ""))
    except ValueError:
        return None


df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
df = df.dropna(subset=["price_clean"]).copy()
print("after price parse:", df.shape)


def parse_area(x):
    """'1200 sqft' -> 1200.0   '140 sqm' -> 1506.96 (sqft)"""
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        num = float("".join(ch for ch in x if ch.isdigit() or ch == "."))
    except ValueError:
        return None
    if "sqm" in x:
        return num * 10.764
    return num


df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area)
df["super_area_sqft"] = df["Super Area"].apply(parse_area)
# fall back to Super Area when Carpet Area is missing
df["carpet_area_sqft"] = df["carpet_area_sqft"].fillna(df["super_area_sqft"])


def parse_floor(x):
    """'3 out of 10' -> 3   'Ground out of 5' -> 0   'Basement' -> -1"""
    if not isinstance(x, str):
        return None
    first = x.split("out of")[0].strip().lower()
    if first in ("ground", "g"):
        return 0
    if "basement" in first:
        return -1
    try:
        return float(first)
    except ValueError:
        return None


df["floor_num"] = df["Floor"].apply(parse_floor)

for col in ["Bathroom", "Balcony"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Car Parking"] = pd.to_numeric(
    df["Car Parking"].astype(str).str.extract(r"(\d+)")[0], errors="coerce"
).fillna(0)

TOP_N_LOCATIONS = 50
top_locations = df["location"].value_counts().nlargest(TOP_N_LOCATIONS).index
df["location_grouped"] = df["location"].where(df["location"].isin(top_locations), "other")

df = df.drop(columns=["Index", "Title", "Description", "Dimensions", "Plot Area"], errors="ignore")

# outlier removal on price-per-sqft
df["price_per_sqft"] = df["price_clean"] / df["carpet_area_sqft"]
p1, p99 = df["price_per_sqft"].quantile([0.01, 0.99])
before = len(df)
df = df[(df["price_per_sqft"].isna()) | ((df["price_per_sqft"] >= p1) & (df["price_per_sqft"] <= p99))]
print(f"outlier removal: {before} -> {len(df)}")

# drop rows missing the core numeric predictor
df = df.dropna(subset=["carpet_area_sqft"])
print("final shape:", df.shape)

# ---------------------------------------------------------------------------
# 2.2 EDA plots (saved as PNG for the notebook / README)
# ---------------------------------------------------------------------------
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(df["price_clean"], log_scale=True, ax=ax)
ax.set_title("Price distribution (log scale)")
ax.set_xlabel("Price (INR)")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/eda_price_dist.png", dpi=110)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
sample = df.sample(min(5000, len(df)), random_state=RANDOM_STATE)
sns.scatterplot(data=sample, x="carpet_area_sqft", y="price_clean", alpha=0.3, ax=ax)
ax.set_yscale("log")
ax.set_xlim(0, sample["carpet_area_sqft"].quantile(0.99))
ax.set_title("Price vs. carpet area")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/eda_price_vs_area.png", dpi=110)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
top15 = (
    df.groupby("location")["price_clean"].mean().sort_values(ascending=False).head(15)
)
sns.barplot(x=top15.values, y=top15.index, ax=ax, color="#4C72B0")
ax.set_xlabel("Average price (INR)")
ax.set_title("Average price by top-15 locations")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/eda_top_locations.png", dpi=110)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="Furnishing", y="price_clean", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Price by furnishing status")
axes[0].tick_params(axis="x", rotation=20)
bath_sub = df[df["Bathroom"].fillna(0) <= 5]
sns.boxplot(data=bath_sub, x="Bathroom", y="price_clean", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Price by number of bathrooms")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/eda_furnishing_bathroom.png", dpi=110)
plt.close(fig)

# ---------------------------------------------------------------------------
# 2.4 Pipeline & Train
# ---------------------------------------------------------------------------
# NOTE ON SAMPLE SIZE: the cleaned dataset has ~175k rows. All cleaning and
# EDA above ran on the FULL dataset. For the actual model fit we take a
# random sample of 50,000 rows -- this keeps training time reasonable on
# modest single-core hardware while still giving the models plenty of data
# to learn from. Remove the .sample(...) call below to train on all rows.
MODEL_SAMPLE_SIZE = 50_000
df_model = df.sample(n=min(MODEL_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

numeric_features = ["carpet_area_sqft", "floor_num", "Bathroom", "Balcony", "Car Parking"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing"]

preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

X = df_model[numeric_features + categorical_features]
y = df_model["price_clean"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Price is heavily right-skewed, so every regressor is wrapped in a
# TransformedTargetRegressor that fits on log1p(y) internally and inverts
# with expm1() automatically. This is a *built-in* sklearn class (unlike a
# hand-rolled wrapper), so the exported pipeline unpickles cleanly in the
# FastAPI backend with nothing but scikit-learn installed.
models = {
    "LinearRegression": TransformedTargetRegressor(
        regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1
    ),
    "RandomForest": TransformedTargetRegressor(
        regressor=RandomForestRegressor(
            n_estimators=120, max_depth=16, n_jobs=-1, random_state=RANDOM_STATE
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
    "GradientBoosting": TransformedTargetRegressor(
        regressor=GradientBoostingRegressor(
            n_estimators=150, max_depth=3, random_state=RANDOM_STATE
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
}

results = {}
fitted = {}
for name, reg in models.items():
    pipe = Pipeline([("prep", preprocessor), ("reg", reg)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = root_mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    fitted[name] = pipe
    print(f"{name:16s} MAE={mae:,.0f}  RMSE={rmse:,.0f}  R2={r2:.4f}")

results_df = pd.DataFrame(results).T.sort_values("R2", ascending=False)
print(results_df)

best_name = results_df.index[0]
best_model = fitted[best_name]
print("winner:", best_name)

# predicted vs actual plot for the winning model
best_pred = best_model.predict(X_test)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_pred, alpha=0.25, s=10)
lims = [0, min(y_test.max(), best_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Actual price")
ax.set_ylabel("Predicted price")
ax.set_title(f"Predicted vs Actual ({best_name})")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/eda_pred_vs_actual.png", dpi=110)
plt.close(fig)

# bonus: 5-fold CV (on log target, R2 scoring)
if best_name == "RandomForest":
    cv_estimator = Pipeline(
        [
            ("prep", preprocessor),
            (
                "reg",
                RandomForestRegressor(
                    n_estimators=60, max_depth=14, n_jobs=-1, random_state=RANDOM_STATE
                ),
            ),
        ]
    )
else:
    cv_estimator = fitted[best_name]

# إضافة تعريف y_train_log هنا
y_train_log = np.log1p(y_train)

cv_scores = cross_val_score(cv_estimator, X_train, y_train_log, cv=5, scoring="r2")
print("5-fold CV R2:", cv_scores, "mean:", cv_scores.mean())
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# 2.6 Export
# ---------------------------------------------------------------------------
# Save best_model directly because TransformedTargetRegressor already handles
# the log1p transformation and inverse expm1() automatically.

joblib.dump(best_model, f"{OUT_DIR}/house_price.pkl")

# sanity check: reload and predict one sample
loaded = joblib.load(f"{OUT_DIR}/house_price.pkl")
sample = X_test.iloc[[0]]
print("reloaded prediction:", loaded.predict(sample)[0], "actual:", y_test.iloc[0])

json.dump(
    sorted(df["location_grouped"].unique().tolist()),
    open(f"{OUT_DIR}/locations.json", "w"),
)

import sklearn

metrics_out = {
    "winner": best_name,
    "sklearn_version": sklearn.__version__,
    "n_rows_final": int(len(df)),
    "results": results,
    "cv_r2_mean": float(cv_scores.mean()),
    "cv_r2_scores": cv_scores.tolist(),
}
json.dump(metrics_out, open(f"{OUT_DIR}/metrics.json", "w"), indent=2)
print("DONE")

shape: (187531, 21)
Index                  int64
Title                    str
Description              str
Amount(in rupees)        str
Price (in rupees)    float64
location                 str
Carpet Area              str
Status                   str
Floor                 object
Transaction              str
Furnishing               str
facing                   str
overlooking              str
Society                  str
Bathroom              object
Balcony               object
Car Parking              str
Ownership                str
Super Area               str
Dimensions           float64
Plot Area            float64
dtype: object
Plot Area            1.000000
Dimensions           1.000000
Society              0.584853
Super Area           0.574225
Car Parking          0.551146
overlooking          0.434254
Carpet Area          0.430185
facing               0.374514
Ownership            0.349366
Balcony              0.260944
Price (in rupees)    0.094198
Floor                0.0377